

---


# <h1 align=center> **Sistema de recomendacion de peliculas** </h1>




---


# <h1 align=center> **Modelo de Machine Learning** </h1>




---



Se trabajará con los siguientes métodos y técnicas de Procesamiento de Lenguaje Natural:

TF-IDF (Frecuencia de Término - Frecuencia Inversa de Documento): Este enfoque transforma el texto en una representación numérica, creando una matriz de características donde cada palabra está asociada a un valor que indica su relevancia en el documento frente al conjunto de documentos (corpus). Estos valores se calculan únicamente a partir de la frecuencia de las palabras en el texto.

Similitud del Coseno: Esta técnica evalúa la similitud entre dos vectores generados a partir del TF-IDF, midiendo qué tan cercanos son en un espacio vectorial. La similitud se basa exclusivamente en los términos y sus ponderaciones en el texto, sin verse influenciada por otros valores numéricos externos.

##Ingesta de datos luego de aplicado el EDA

In [2]:
#Importar librerías
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from wordcloud import WordCloud, STOPWORDS
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
nltk.download('punkt')
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ruth\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ruth\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
# Cargar el DataFrame que se ha dividido en dos partes para poderlo subir a GitHub
parte1 = pd.read_csv('data_preparada_parte1.csv')
parte2 = pd.read_csv('data_preparada_parte2.csv')

# Concatenar el dataset en un DataFrame
df =pd.concat([parte1,parte2], ignore_index = True)

In [4]:
df.dtypes

id                     int64
title                 object
tagline               object
overview              object
collection            object
genre                 object
company               object
original_language     object
runtime              float64
popularity           float64
vote_count           float64
vote_average         float64
release_date          object
release_year           int64
status                object
country               object
language              object
revenue              float64
budget               float64
return               float64
actor                 object
director              object
dtype: object

In [5]:
df['title'][:30]

0                          Toy Story
1                            Jumanji
2                   Grumpier Old Men
3                  Waiting to Exhale
4        Father of the Bride Part II
5                               Heat
6                            Sabrina
7                       Tom and Huck
8                       Sudden Death
9                          GoldenEye
10            The American President
11       Dracula: Dead and Loving It
12                             Balto
13                             Nixon
14                  Cutthroat Island
15                            Casino
16             Sense and Sensibility
17                        Four Rooms
18    Ace Ventura: When Nature Calls
19                       Money Train
20                        Get Shorty
21                           Copycat
22                         Assassins
23                            Powder
24                 Leaving Las Vegas
25                           Othello
26                      Now and Then
2

In [6]:
df.columns

Index(['id', 'title', 'tagline', 'overview', 'collection', 'genre', 'company',
       'original_language', 'runtime', 'popularity', 'vote_count',
       'vote_average', 'release_date', 'release_year', 'status', 'country',
       'language', 'revenue', 'budget', 'return', 'actor', 'director'],
      dtype='object')

In [7]:
#revisar valores nulos
df.isnull().sum()

id                       0
title                    0
tagline              24959
overview               941
collection           40968
genre                 2384
company              11847
original_language       11
runtime                246
popularity               0
vote_count               0
vote_average             0
release_date             0
release_year             0
status                  80
country               6209
language              3889
revenue                  0
budget                   0
return                   0
actor                 2359
director              1007
dtype: int64

In [8]:
df.shape

(45346, 22)

**Se crearán dos columnas como resultado del análisis efectuado en el EDA**

In [9]:
df['title'][:3]

0           Toy Story
1             Jumanji
2    Grumpier Old Men
Name: title, dtype: object

In [10]:
# Extraer el primer actor de la lista en la columna 'actors'
df['first_actor'] = df['actor'].apply(lambda x: x.split(',')[0] if pd.notna(x) else '')

# Extraer el primer director de la lista en la columna 'director'
df['first_director'] = df['director'].apply(lambda x: x.split(',')[0] if pd.notna(x) else '')

In [11]:
df.head()

,id,title,tagline,overview,collection,genre,company,original_language,runtime,popularity,...,status,country,language,revenue,budget,return,actor,director,first_actor,first_director
0,862,Toy Story,NaN,"Led by Woody, Andy's toys live happily in his ...",Toy Story Collection,"Animation, Comedy, Family",Pixar Animation Studios,en,81.0,21.946943,...,Released,United States of America,English,373554033.0,30000000.0,12.45,"Tom Hanks, Tim Allen, Don Rickles, Jim Varney,...",John Lasseter,Tom Hanks,John Lasseter
1,8844,Jumanji,Roll the dice and unleash the excitement!,When siblings Judy and Peter discover an encha...,NaN,"Adventure, Fantasy, Family","TriStar Pictures, Teitler Film, Interscope Com...",en,104.0,17.015539,...,Released,United States of America,"English, Français",262797249.0,65000000.0,4.04,"Robin Williams, Jonathan Hyde, Kirsten Dunst, ...",Joe Johnston,Robin Williams,Joe Johnston
2,15602,Grumpier Old Men,Still Yelling. Still Fighting. Still Ready for...,A family wedding reignites the ancient feud be...,Grumpy Old Men Collection,"Romance, Comedy","Warner Bros., Lancaster Gate",en,101.0,11.712900,...,Released,United States of America,English,0.0,0.0,0.00,"Walter Matthau, Jack Lemmon, Ann-Margret, Soph...",Howard Deutch,Walter Matthau,Howard Deutch
3,31357,Waiting to Exhale,Friends are the people who let you be yourself...,"Cheated on, mistreated and stepped on, the wom...",NaN,"Comedy, Drama, Romance",Twentieth Century Fox Film Corporation,en,127.0,3.859495,...,Released,United States of America,English,81452156.0,16000000.0,5.09,"Whitney Houston, Angela Bassett, Loretta Devin...",Forest Whitaker,Whitney Houston,Forest Whitaker
4,11862,Father of the Bride Part II,Just When His World Is Back To Normal... He's ...,Just when George Banks has recovered from his ...,Father of the Bride Collection,Comedy,"Sandollar Productions, Touchstone Pictures",en,106.0,8.387519,...,Released,United States of America,English,76578911.0,0.0,0.00,"Steve Martin, Diane Keaton, Martin Short, Kimb...",Charles Shyer,Steve Martin,Charles Shyer


In [12]:
df.columns

Index(['id', 'title', 'tagline', 'overview', 'collection', 'genre', 'company',
       'original_language', 'runtime', 'popularity', 'vote_count',
       'vote_average', 'release_date', 'release_year', 'status', 'country',
       'language', 'revenue', 'budget', 'return', 'actor', 'director',
       'first_actor', 'first_director'],
      dtype='object')

Se utilizará el atributo overview en combinación de otros más para evaluar los distintos modelos

In [13]:
df = df.fillna('')  # Reemplazar nulos con cadenas vacias

##**Model1**

In [14]:
#Extrar las columnas relevantes para el modelo
model1 = df[['title', 'overview']].copy()

In [15]:
# Crear una instancia de TfidfVectorizer
tfidf_1 = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))

In [16]:
# Aplicar la transformación TF-IDF al contenido
tfidf_matriz_1 = tfidf_1.fit_transform(model1['overview'])


In [17]:
# Para visualizar como funciona la transformacion TF-IDF
# Convertimos solo las primeras 5 filas de la matriz TF-IDF a un array denso para inspección
partial_tfidf_df = pd.DataFrame(tfidf_matriz_1[:5].toarray(), columns=tfidf_1.get_feature_names_out())

# Mostrar las primeras filas del DataFrame
partial_tfidf_df.head()


,00,00 agent,00 body,00 editor,00 foot,00 furnish,00 middle,00 pm,00 rescue,00 schneider,...,주식회사,주식회사 means,찾기,찾기 주식회사,첫사랑,첫사랑 찾기,ﬁrst,ﬁrst rwandan,ﬁve,ﬁve friends
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
# Función para obtener recomendaciones
def recomendacion_m1(titulo):
    #Crear una serie que asigna un índice a cada título de las películas
    indices = pd.Series(model1.index, index=model1['title']).drop_duplicates()
    if titulo not in indices:
        return 'La película ingresada no se encuentra en la base de datos'
    else:
        #Obtener el índice de la película que coincide con el título
        ind = pd.Series(indices[titulo]) if titulo in indices else None
        #Si el título de la película está duplicado, devolver el índice de la primera aparición del título en el DataFrame
        if model1.duplicated(['title']).any():
            primer_ind = model1[model1['title'] == titulo].index[0]
            if not ind.equals(pd.Series(primer_ind)):
                ind = pd.Series(primer_ind)
        #Calcular la similitud coseno entre la película de entrada y todas las demás películas en la matriz de características
        cosine_sim = cosine_similarity(tfidf_matriz_1[ind], tfidf_matriz_1).flatten()
        simil = sorted(enumerate(cosine_sim), key=lambda x: x[1], reverse=True)[1:6]
        #Verificar que los índices obtenidos son válidos
        valid_ind = [i[0] for i in simil if i[0] < len(model1)]
         
      #Obtener los títulos de las películas más similares utilizando el índice de cada película
        recomendaciones = model1.iloc[valid_ind]['title'].tolist()
       # Devolver la lista de títulos de las películas recomendadas
        return recomendaciones

##**Se realizan distintas pruebas para evaluar el modelo**

In [19]:

recomendacion_m1('The Dark Knight Rises')

['Batman Forever',
 'The Dark Knight',
 'Batman',
 'Batman Returns',
 'Batman: Under the Red Hood']

In [108]:

recomendacion_m1('Jumanji')

['Word Wars', 'The Bar', 'Table No. 21', 'Quintet', 'The Dark Angel']

In [109]:

recomendacion_m1('Titanic')

['Grantham and Rose',
 'The Legend of 1900',
 'Raise the Titanic',
 'Dustbin Baby',
 'The Experience']

In [110]:

recomendacion_m1('Wonder Woman')

['Wide Eyed and Legless',
 'Trevor Noah: The Daywalker',
 'Nightbeast',
 'The Machinist',
 'Escape from New York']

In [111]:

recomendacion_m1('Star Wars')

['The Empire Strikes Back',
 'The Star Wars Holiday Special',
 'Star Wars: The Force Awakens',
 'Return of the Jedi',
 'Samson and the Seven Miracles of the World']

In [21]:

recomendacion_m1('The Avengers')

['The Work and the Glory',
 'Monty Python and the Holy Grail',
 "Sir Arne's Treasure",
 'Prince of Persia: The Sands of Time',
 'Doctor in Distress']

##**Model2**

In [20]:
#Extrar las columnas relevantes para el modelo
model2 = df[['title', 'overview', 'first_director']].copy()

In [21]:
# Crear una instancia de TfidfVectorizer
tfidf_2 = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))

In [22]:
model2['concatenado'] = model2['overview'] + ' ' + model2['first_director']

In [23]:
# Aplicar la transformación TF-IDF
tfidf_matriz_2 = tfidf_2.fit_transform(model2['concatenado'])

In [24]:
# Función para obtener recomendaciones
def recomendacion_m2(titulo):
    #Crear una serie que asigna un índice a cada título de las películas
    indices = pd.Series(model2.index, index=model2['title']).drop_duplicates()
    if titulo not in indices:
        return 'La película ingresada no se encuentra en la base de datos'
    else:
        #Obtener el índice de la película que coincide con el título
        ind = pd.Series(indices[titulo]) if titulo in indices else None
        #Si el título de la película está duplicado, devolver el índice de la primera aparición del título en el DataFrame
        if model2.duplicated(['title']).any():
            primer_ind = model2[model1['title'] == titulo].index[0]
            if not ind.equals(pd.Series(primer_ind)):
                ind = pd.Series(primer_ind)
        #Calcular la similitud coseno entre la película de entrada y todas las demás películas en la matriz de características
        cosine_sim = cosine_similarity(tfidf_matriz_2[ind], tfidf_matriz_2).flatten()
        simil = sorted(enumerate(cosine_sim), key=lambda x: x[1], reverse=True)[1:6]
        #Verificar que los índices obtenidos son válidos
        valid_ind = [i[0] for i in simil if i[0] < len(model2)]
        #Obtener los títulos de las películas más similares utilizando el índice de cada película
        recomendaciones = model2.iloc[valid_ind]['title'].tolist()
        #Devolver la lista de títulos de las películas recomendadas
        return recomendaciones


In [25]:
recomendacion_m2('Titanic')

['Grantham and Rose',
 'Ghosts of the Abyss',
 'The Legend of 1900',
 'Avatar 2',
 'Dustbin Baby']

In [35]:
recomendacion_m2('Avatar')

['Aliens of the Deep',
 'Avatar 2',
 'Bloodbrothers',
 'Stand by Me Doraemon',
 'Ghosts of the Abyss']

In [36]:
recomendacion_m2('Wonder Woman')

['Wide Eyed and Legless',
 'Escape from New York',
 'The Machinist',
 'Green Lantern: First Flight',
 'Our Dancing Daughters']

In [37]:
recomendacion_m2('Star Wars')

['The Empire Strikes Back',
 'The Star Wars Holiday Special',
 'Star Wars: The Force Awakens',
 'Return of the Jedi',
 'Samson and the Seven Miracles of the World']

In [38]:
recomendacion_m2('The Avengers')

['The Right Kind of Wrong',
 'The Work and the Glory',
 'Benny & Joon',
 'Monty Python and the Holy Grail',
 "Sir Arne's Treasure"]

In [39]:
recomendacion_m2('Shrek')

['Shrek 2',
 'Shrek the Third',
 'Silk Stockings',
 'Scared Shrekless',
 'Shrek Forever After']

##**Model3**

In [26]:
#Extrar las columnas relevantes para el modelo
model3 = df[['id','title', 'overview', 'genre', 'first_director']].copy()

In [27]:
##Se separan los géneros y se convierten en palabras individuales
model3['genre'] = model3['genre'].fillna('').apply(lambda x: ' '.join(x.replace(',', ' ').replace('-', '').lower().split()))

In [28]:
# Se crea una instancia de la clase TfidfVectorizer
tfidf_3 = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
# Aplicar la transformación TF-IDF al texto contenido en las columnas "overview_clean", "genres" y "director" del dataframe 'modelo4'
tfidf_matriz_3 = tfidf_3.fit_transform(model3['overview'] + ' ' + model3['genre'] + ' ' + model3['first_director'])

In [29]:
tfidf_matriz_3.shape

(45346, 1107314)

In [30]:
# Función para obtener recomendaciones
def recomendacion_m3(titulo):
    #Crear una serie que asigna un índice a cada título de las películas
    indices = pd.Series(model3.index, index=model3['title']).drop_duplicates()
    if titulo not in indices:
        return 'La película ingresada no se encuentra en la base de datos'
    else:
        #Obtener el índice de la película que coincide con el título
        ind = pd.Series(indices[titulo]) if titulo in indices else None
        #Si el título de la película está duplicado, devolver el índice de la primera aparición del título en el DataFrame
        if model1.duplicated(['title']).any():
            primer_ind = model3[model3['title'] == titulo].index[0]
            if not ind.equals(pd.Series(primer_ind)):
                ind = pd.Series(primer_ind)
        #Calcular la similitud coseno entre la película de entrada y todas las demás películas en la matriz de características
        cosine_sim = cosine_similarity(tfidf_matriz_3[ind], tfidf_matriz_3).flatten()
        simil = sorted(enumerate(cosine_sim), key=lambda x: x[1], reverse=True)[1:6]
        #Verificar que los índices obtenidos son válidos
        valid_ind = [i[0] for i in simil if i[0] < len(model3)]
        #Obtener los títulos de las películas más similares utilizando el índice de cada película
        recomendaciones = model3.iloc[valid_ind]['title'].tolist()
        #Devolver la lista de títulos de las películas recomendadas
        return recomendaciones

In [31]:
recomendacion_m3('Titanic')

['Grantham and Rose',
 'Ghosts of the Abyss',
 'The Legend of 1900',
 'Dustbin Baby',
 'Raise the Titanic']

In [56]:
recomendacion_m3('Avatar')

['Avatar 2',
 'Aliens of the Deep',
 'Bloodbrothers',
 'Stand by Me Doraemon',
 'The War of the Robots']

In [57]:
recomendacion_m3('Wonder Woman')

['DC Showcase: Catwoman',
 'Green Lantern: First Flight',
 'Green Lantern: Emerald Knights',
 'Superman: Doomsday',
 'Wide Eyed and Legless']

In [58]:
recomendacion_m3('Star Wars')

['The Empire Strikes Back',
 'The Star Wars Holiday Special',
 'Star Wars: The Force Awakens',
 'Return of the Jedi',
 'Star Wars: Episode I - The Phantom Menace']

In [43]:
recomendacion_m3('The Avengers')

['The Work and the Glory',
 'The Right Kind of Wrong',
 'Benny & Joon',
 'Monty Python and the Holy Grail',
 'Diabolique']

##**Se decide probar con los atributos 'title', 'genre', 'first_actor', 'first_director' esperando tener una posible mejora en cuanto a la precisión de las recomendaciones**

##**Model4**

In [32]:
#Extrar las columnas relevantes para el modelo
model4 = df[['id','title', 'genre', 'first_actor', 'first_director']].copy()

In [33]:
#Se separan los géneros y se convierten en palabras individuales
model4['genre'] = model4['genre'].fillna('').apply(lambda x: ' '.join(x.replace(',', ' ').replace('-', '').lower().split()))

In [34]:
#Se crea una instancia de la clase TfidfVectorizer
tfidf_4 = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
# Aplicar la transformación TF-IDF al texto contenido en las columnas "overview_clean", "genres" y "director" del dataframe 'modelo4'
tfidf_matriz_4 = tfidf_4.fit_transform(model4['genre'] + ' ' + model4['first_actor'] + ' ' + model4['first_director'])

In [35]:
# Función para obtener recomendaciones
def recomendacion_m4(titulo):
    #Crear una serie que asigna un índice a cada título de las películas
    indices = pd.Series(model4.index, index=model4['title']).drop_duplicates()
    if titulo not in indices:
        return 'La película ingresada no se encuentra en la base de datos'
    else:
        #Obtener el índice de la película que coincide con el título
        ind = pd.Series(indices[titulo]) if titulo in indices else None
        #Si el título de la película está duplicado, devolver el índice de la primera aparición del título en el DataFrame
        if model4.duplicated(['title']).any():
            primer_ind = model4[model4['title'] == titulo].index[0]
            if not ind.equals(pd.Series(primer_ind)):
                ind = pd.Series(primer_ind)
        #Calcular la similitud coseno entre la película de entrada y todas las demás películas en la matriz de características
        cosine_sim = cosine_similarity(tfidf_matriz_4[ind], tfidf_matriz_4).flatten()
        simil = sorted(enumerate(cosine_sim), key=lambda x: x[1], reverse=True)[1:6]
        #Verificar que los índices obtenidos son válidos
        valid_ind = [i[0] for i in simil if i[0] < len(model4)]
        #Obtener los títulos de las películas más similares utilizando el índice de cada película
        recomendaciones = model4.iloc[valid_ind]['title'].tolist()
        #Devolver la lista de títulos de las películas recomendadas
        return recomendaciones

In [36]:
recomendacion_m4('Titanic')

['Avatar 2', 'Jude', 'Sense and Sensibility', 'A Little Chaos', 'The Reader']

In [70]:
recomendacion_m4('Avatar')

['Avatar 2',
 'Clash of the Titans',
 'Wrath of the Titans',
 "The Hunter's Prayer",
 'The Shack']

In [71]:
recomendacion_m4('Shrek')

['Shrek 2',
 'Shrek the Third',
 'Shrek the Halls',
 'The Chronicles of Narnia: Prince Caspian',
 'The Chronicles of Narnia: The Lion, the Witch and the Wardrobe']

In [72]:
recomendacion_m4('Star Wars')

['Return of the Jedi',
 'The Empire Strikes Back',
 'Comic Book: The Movie',
 'Time Runner',
 'Star Wars: Episode I - The Phantom Menace']

In [73]:
recomendacion_m4('Toy Story')

['Toy Story 2', 'Luxo Jr.', 'Tin Toy', "Red's Dream", 'Knick Knack']

In [74]:
recomendacion_m4('Star Trek')

['Star Trek Into Darkness',
 'Star Trek Beyond',
 'Blind Dating',
 'Jack Ryan: Shadow Recruit',
 'People Like Us']

In [75]:
recomendacion_m4('Mars')

['Blazing a Trail to the Stars',
 'White Out, Black In',
 'I Know What I Saw',
 'Infinite',
 'There Will Come Soft Rains']

In [76]:
recomendacion_m4('Avatar')

['Avatar 2',
 'Clash of the Titans',
 'Wrath of the Titans',
 "The Hunter's Prayer",
 'The Shack']

In [77]:
recomendacion_m4('The Hunger Games')

['The Hunger Games: Mockingjay - Part 1',
 'The Hunger Games: Mockingjay - Part 2',
 'The Hunger Games: Catching Fire',
 'Passengers',
 'Joy']

In [78]:
recomendacion_m4('Forrest Gump')

['Cast Away',
 'The Polar Express',
 'Larry Crowne',
 'Sleepless in Seattle',
 'The Money Pit']

In [79]:
recomendacion_m4('Men in Black')

['Men in Black II',
 'The Sunset Limited',
 'The Good Old Boys',
 'The Three Burials of Melquiades Estrada',
 'Wild Wild West']

In [80]:
recomendacion_m4('Wonder Woman')

['Green Lantern: Emerald Knights',
 'DC Showcase: Catwoman',
 'Superman: Doomsday',
 'Green Lantern: First Flight',
 'Justice League: Crisis on Two Earths']

##**Model5**

In [45]:
#Extrar las columnas relevantes para el modelo
model5 = df[['id','title', 'genre', 'tagline' ,'first_actor', 'first_director']].copy()

In [46]:
#Se separan los géneros y se convierten en palabras individuales
model5['genre'] = model5['genre'].fillna('').apply(lambda x: ' '.join(x.replace(',', ' ').replace('-', '').lower().split()))
#Se separan los slogans y se convierten en palabras individuales
model5['tagline'] = model5['tagline'].fillna('').apply(lambda x: ' '.join(x.replace(',', ' ').replace('-', '').lower().split()))
#Se crea una instancia de la clase TfidfVectorizer
tfidf_5 = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
#Aplicar la transformación TF-IDF y obtener matriz numérica
tfidf_matriz_5 = tfidf_5.fit_transform(model5['genre'] + ' ' + model5['tagline'] + ' ' + model5['first_actor']+ ' ' + model5['first_director'])

In [47]:
# Función para obtener recomendaciones
def recomendacion_m5(titulo):
  #Crear una serie que asigna un índice a cada título de las películas
    indices = pd.Series(model5.index, index=model5['title']).drop_duplicates()
    if titulo not in indices:
        return 'La película ingresada no se encuentra en la base de datos'
    else:
        #Obtener el índice de la película que coincide con el título
        ind = pd.Series(indices[titulo]) if titulo in indices else None
        #Si el título de la película está duplicado, devolver el índice de la primera aparición del título en el DataFrame
        if model5.duplicated(['title']).any():
            primer_ind = model5[model5['title'] == titulo].index[0]
            if not ind.equals(pd.Series(primer_ind)):
                ind = pd.Series(primer_ind)
        #Calcular la similitud coseno entre la película de entrada y todas las demás películas en la matriz de características
        cosine_sim = cosine_similarity(tfidf_matriz_5[ind], tfidf_matriz_5).flatten()
        simil = sorted(enumerate(cosine_sim), key=lambda x: x[1], reverse=True)[1:6]
        #Verificar que los índices obtenidos son válidos
        valid_ind = [i[0] for i in simil if i[0] < len(model5)]
        #Obtener los títulos de las películas más similares utilizando el índice de cada película
        recomendaciones = model5.iloc[valid_ind]['title'].tolist()
        #Devolver la lista de títulos de las películas recomendadas
        return recomendaciones

In [48]:
recomendacion_m5('Toy Story')

['Toy Story 2', 'Luxo Jr.', 'Tin Toy', "Red's Dream", 'Knick Knack']

In [60]:
recomendacion_m5('Wonder Woman')

['DC Showcase: Catwoman',
 'Green Lantern: Emerald Knights',
 'There Will Come Soft Rains',
 'Superman/Batman: Apocalypse',
 'Superman: Doomsday']

In [61]:
recomendacion_m5('Avatar')

['Avatar 2',
 'Clash of the Titans',
 'The Shack',
 'T2 3-D: Battle Across Time',
 "The Hunter's Prayer"]

In [91]:
recomendacion_m5('Batman Returns')

['The Merry Gentleman',
 'Hansel and Gretel',
 'The Last Time',
 'Dark Shadows',
 'Live from Baghdad']

In [92]:
recomendacion_m5('Star Wars')

['The Star Wars Holiday Special',
 'The Galaxy Invader',
 'Comic Book: The Movie',
 "On Her Majesty's Secret Service",
 'The Empire Strikes Back']

In [93]:
recomendacion_m5('The Avengers')

['The Invisible Woman',
 'Onegin',
 'Get Smart',
 'A Dangerous Man: Lawrence After Arabia',
 'Tall Tale']

In [94]:
recomendacion_m5('The Dark Knight Rises')

['The Dark Knight', 'Batman Begins', 'Following', 'Harsh Times', 'Last Exit']

In [95]:
recomendacion_m5('Jumanji')

['Seize the Day',
 "Fathers' Day",
 'Hook',
 'The World According to Garp',
 'The Pagemaster']

In [96]:
recomendacion_m5('Titanic')

['Avatar 2',
 'Little Children',
 'Deep Sea 3D',
 'Christmas Carol: The Movie',
 'Sense and Sensibility']

In [97]:
recomendacion_m5('Men in Black')

['Men in Black II',
 'The Good Old Boys',
 'The Sunset Limited',
 'The Three Burials of Melquiades Estrada',
 'Men in Black 3']

##**Modelo seleccionado**
El modelo Model3 fue el que presentó una mejor recomendación pero tiene un mayor costo computacional. Es por ello que elegimos el segundo mejor modelo, Model5, que podrá adaptarse a las limitaciones de memoria del sitio render.

In [49]:
# Se extraen las columnas necesarias para el Sistema de Recomendación
model5 = df[['id','title', 'genre', 'tagline' ,'first_actor', 'first_director']]

In [50]:
# Se exportan los datos a formato csv
model5.to_csv('movies_model5.csv')

In [51]:
model5.shape

(45346, 6)

In [52]:
# Conslidamos todas las recomendaciones por pelicula
import pandas as pd
# Crear un DataFrame vacío para almacenar las recomendaciones
recomendaciones_df = pd.DataFrame(columns=['Pelicula', 'Recomendacion', 'Modelo'])

# Ingresar la pelicula a consultar
# Lista de títulos de películas para las que se desean obtener recomendaciones
titulos_peliculas = ['Pocahontas','Jumanji','Balto','Clueless','Avatar','Titanic','Bad Boys','Billy Madison','Pulp Fiction','Selena']

# Iterar sobre los títulos de películas
for titulo in titulos_peliculas:
  # Obtener recomendaciones para cada modelo
  recomendaciones_m1 = recomendacion_m1(titulo)
  recomendaciones_m2 = recomendacion_m2(titulo)
  recomendaciones_m3 = recomendacion_m3(titulo)
  recomendaciones_m4 = recomendacion_m4(titulo)
  recomendaciones_m5 = recomendacion_m5(titulo)

  # Agregar las recomendaciones al DataFrame
  if isinstance(recomendaciones_m1, list):
    for recomendacion in recomendaciones_m1:
      recomendaciones_df = pd.concat([recomendaciones_df, pd.DataFrame({'Pelicula': titulo, 'Recomendacion': recomendacion, 'Modelo': 'Modelo 1'}, index=[0])], ignore_index=True)
  if isinstance(recomendaciones_m2, list):
    for recomendacion in recomendaciones_m2:
      recomendaciones_df = pd.concat([recomendaciones_df, pd.DataFrame({'Pelicula': titulo, 'Recomendacion': recomendacion, 'Modelo': 'Modelo 2'}, index=[0])], ignore_index=True)
  if isinstance(recomendaciones_m3, list):
    for recomendacion in recomendaciones_m3:
      recomendaciones_df = pd.concat([recomendaciones_df, pd.DataFrame({'Pelicula': titulo, 'Recomendacion': recomendacion, 'Modelo': 'Modelo 3'}, index=[0])], ignore_index=True)
  if isinstance(recomendaciones_m4, list):
    for recomendacion in recomendaciones_m4:
      recomendaciones_df = pd.concat([recomendaciones_df, pd.DataFrame({'Pelicula': titulo, 'Recomendacion': recomendacion, 'Modelo': 'Modelo 4'}, index=[0])], ignore_index=True)
  if isinstance(recomendaciones_m5, list):
    for recomendacion in recomendaciones_m5:
      recomendaciones_df = pd.concat([recomendaciones_df, pd.DataFrame({'Pelicula': titulo, 'Recomendacion': recomendacion, 'Modelo': 'Modelo 5'}, index=[0])], ignore_index=True)


# Mostrar el DataFrame con las recomendaciones
recomendaciones_df.head()

,Pelicula,Recomendacion,Modelo
0,Pocahontas,Just Friends,Modelo 1
1,Pocahontas,Doctor Who: Time Crash,Modelo 1
2,Pocahontas,Black Eye,Modelo 1
3,Pocahontas,Clouds of Sils Maria,Modelo 1
4,Pocahontas,Lizzie,Modelo 1


In [153]:
recomendaciones_df.shape

(250, 3)

In [53]:
# Seleccionar solo las dos primeras columnas
df_simplificado = recomendaciones_df[['Pelicula', 'Recomendacion']]

# Eliminar filas duplicadas
df_simplificado = df_simplificado.drop_duplicates()

# Mostrar el resultado
print(df_simplificado)


       Pelicula              Recomendacion
0    Pocahontas               Just Friends
1    Pocahontas     Doctor Who: Time Crash
2    Pocahontas                  Black Eye
3    Pocahontas       Clouds of Sils Maria
4    Pocahontas                     Lizzie
..          ...                        ...
241      Selena  Why Do Fools Fall In Love
242      Selena                  My Family
243      Selena                     Enough
244      Selena          The Boy Next Door
246      Selena          Maid in Manhattan

[140 rows x 2 columns]


In [157]:
df_simplificado.to_csv('recomendaciones.csv', index=False)


In [54]:

import pandas as pd

def obtener_recomendaciones_para_pelicula(titulo_pelicula):
    # Crear un DataFrame vacío para almacenar las recomendaciones
    recomendaciones_df = pd.DataFrame(columns=['Pelicula', 'Recomendacion', 'Modelo'])

    # Obtener recomendaciones para la película en cada modelo
    recomendaciones_m1 = recomendacion_m1(titulo_pelicula)
    recomendaciones_m2 = recomendacion_m2(titulo_pelicula)
    recomendaciones_m3 = recomendacion_m3(titulo_pelicula)
    recomendaciones_m4 = recomendacion_m4(titulo_pelicula)
    recomendaciones_m5 = recomendacion_m5(titulo_pelicula)

    # Agregar las recomendaciones al DataFrame
    if isinstance(recomendaciones_m1, list):
        for recomendacion in recomendaciones_m1:
            recomendaciones_df = pd.concat([recomendaciones_df, pd.DataFrame({'Pelicula': titulo_pelicula, 'Recomendacion': recomendacion, 'Modelo': 'Modelo 1'}, index=[0])], ignore_index=True)
    if isinstance(recomendaciones_m2, list):
        for recomendacion in recomendaciones_m2:
            recomendaciones_df = pd.concat([recomendaciones_df, pd.DataFrame({'Pelicula': titulo_pelicula, 'Recomendacion': recomendacion, 'Modelo': 'Modelo 2'}, index=[0])], ignore_index=True)
    if isinstance(recomendaciones_m3, list):
        for recomendacion in recomendaciones_m3:
            recomendaciones_df = pd.concat([recomendaciones_df, pd.DataFrame({'Pelicula': titulo_pelicula, 'Recomendacion': recomendacion, 'Modelo': 'Modelo 3'}, index=[0])], ignore_index=True)
    if isinstance(recomendaciones_m4, list):
        for recomendacion in recomendaciones_m4:
            recomendaciones_df = pd.concat([recomendaciones_df, pd.DataFrame({'Pelicula': titulo_pelicula, 'Recomendacion': recomendacion, 'Modelo': 'Modelo 4'}, index=[0])], ignore_index=True)
    if isinstance(recomendaciones_m5, list):
        for recomendacion in recomendaciones_m5:
            recomendaciones_df = pd.concat([recomendaciones_df, pd.DataFrame({'Pelicula': titulo_pelicula, 'Recomendacion': recomendacion, 'Modelo': 'Modelo 5'}, index=[0])], ignore_index=True)

    # Contar las veces que cada película ha sido recomendada y agregar los modelos que lo recomiendan
    recomendaciones_consolidadas_df = recomendaciones_df.groupby(['Recomendacion'])['Modelo'].agg(['nunique', list]).reset_index()
    recomendaciones_consolidadas_df.rename(columns={'nunique': 'n_recomendada', 'list': 'modelos'}, inplace=True)

    # Agregar una columna con el título de la película
    recomendaciones_consolidadas_df['Pelicula'] = titulo_pelicula

    return recomendaciones_consolidadas_df

# Ejemplo de uso para cualquier película
titulo_pelicula = 'Pocahontas' # Cambia este título según la película que quieras consultar
recomendaciones_consolidadas = obtener_recomendaciones_para_pelicula(titulo_pelicula)

# Mostrar el DataFrame con las recomendaciones consolidadas
print(recomendaciones_consolidadas)


                            Recomendacion  n_recomendada  \
0                               Black Eye              3   
1                    Clouds of Sils Maria              3   
2                  Doctor Who: Time Crash              3   
3                    Dragonball Evolution              1   
4                         I Remember Mama              1   
5                            Just Friends              3   
6     Lakota Woman: Siege at Wounded Knee              1   
7                          Les Misérables              1   
8                                  Lizzie              1   
9                                 Lorenzo              4   
10  Pocahontas II: Journey to a New World              2   
11            Songs My Brothers Taught Me              2   

                                     modelos    Pelicula  
0             [Modelo 1, Modelo 2, Modelo 3]  Pocahontas  
1             [Modelo 1, Modelo 2, Modelo 3]  Pocahontas  
2             [Modelo 1, Modelo 2, Modelo 

Se solicitó a chat GPT que se brinde una evaluacion del 1-10 donde se califique cuan recomendable es cada pelicula, para una persona a quien le gustó la pelicul "Avatar" y en el archivo csv se muestran los resultados.

In [55]:
PuntajeIA = pd.read_csv('puntaje_IA_recomendaciones.csv')
print(PuntajeIA)

       Pelicula              Recomendacion  Puntaje
0    Pocahontas               Just Friends        5
1    Pocahontas     Doctor Who: Time Crash        3
2    Pocahontas                  Black Eye        2
3    Pocahontas       Clouds of Sils Maria        7
4    Pocahontas                     Lizzie       10
..          ...                        ...      ...
135      Selena  Why Do Fools Fall In Love       10
136      Selena                  My Family        5
137      Selena                     Enough        5
138      Selena          The Boy Next Door        8
139      Selena          Maid in Manhattan        2

[140 rows x 3 columns]


In [60]:
# Realizar la combinación manteniendo todos los registros de recomendaciones_df
recomendaciones_combinadas = recomendaciones_df.merge(PuntajeIA, on='Recomendacion', how='left')

# Filtrar registros donde las películas coincidan o solo mantener la columna de recomendaciones_df
recomendaciones_combinadas = recomendaciones_combinadas[
    (recomendaciones_combinadas['Pelicula_x'] == recomendaciones_combinadas['Pelicula_y']) | 
    recomendaciones_combinadas['Pelicula_y'].isnull()
]

# Mantener solo una columna 'Pelicula'
recomendaciones_combinadas['Pelicula'] = recomendaciones_combinadas['Pelicula_x']

# Eliminar columnas innecesarias
recomendaciones_combinadas = recomendaciones_combinadas.drop(['Pelicula_x', 'Pelicula_y'], axis=1)

# Mostrar el resultado final
print(recomendaciones_combinadas)



              Recomendacion    Modelo  Puntaje    Pelicula
0              Just Friends  Modelo 1        5  Pocahontas
1    Doctor Who: Time Crash  Modelo 1        3  Pocahontas
2                 Black Eye  Modelo 1        2  Pocahontas
3      Clouds of Sils Maria  Modelo 1        7  Pocahontas
4                    Lizzie  Modelo 1       10  Pocahontas
..                      ...       ...      ...         ...
258              Bordertown  Modelo 5        6      Selena
259       Maid in Manhattan  Modelo 5        2      Selena
260                El Norte  Modelo 5        1      Selena
261                  Enough  Modelo 5        5      Selena
262               My Family  Modelo 5        5      Selena

[250 rows x 4 columns]


In [62]:
# Agrupar por el nombre del modelo y sumar los puntajes
df_puntaje_sumado = recomendaciones_combinadas.groupby('Modelo')['Puntaje'].sum().reset_index()

# Ordenar el dataframe de manera descendente según la columna 'puntaje'
df_puntaje_sumado = df_puntaje_sumado.sort_values(by='Puntaje', ascending=False)

# Mostrar el resultado
print(df_puntaje_sumado)



     Modelo  Puntaje
2  Modelo 3      278
4  Modelo 5      278
1  Modelo 2      277
3  Modelo 4      275
0  Modelo 1      265


In [63]:
# Realizar la suma de puntajes por modelo, agrupando por 'Modelo'
df_puntaje_sumado = recomendaciones_combinadas.groupby('Modelo')['Puntaje'].sum().reset_index()

# Obtener las películas únicas para crear columnas
peliculas_unicas = recomendaciones_combinadas['Pelicula'].unique()

# Iterar sobre las películas únicas y sumar los puntajes para cada una
for pelicula in peliculas_unicas:
    columna_nombre = f'Puntaje_{pelicula}'
    # Sumar puntajes por cada película específica
    df_puntaje_sumado[columna_nombre] = recomendaciones_combinadas[
        recomendaciones_combinadas['Pelicula'] == pelicula
    ].groupby('Modelo')['Puntaje'].sum().reindex(df_puntaje_sumado['Modelo'], fill_value=0).values

# Ordenar el dataframe de manera descendente según la columna 'Puntaje'
df_puntaje_sumado = df_puntaje_sumado.sort_values(by='Puntaje', ascending=False)

# Mostrar el resultado
print(df_puntaje_sumado)


     Modelo  Puntaje  Puntaje_Pocahontas  Puntaje_Jumanji  Puntaje_Balto  \
2  Modelo 3      278                  23               24             34   
4  Modelo 5      278                  24               31             40   
1  Modelo 2      277                  23               27             36   
3  Modelo 4      275                  18               17             37   
0  Modelo 1      265                  27               26             29   

   Puntaje_Clueless  Puntaje_Avatar  Puntaje_Titanic  Puntaje_Bad Boys  \
2                29              43               29                20   
4                32              22               23                32   
1                22              40               27                20   
3                28              30               30                21   
0                22              37               31                20   

   Puntaje_Billy Madison  Puntaje_Pulp Fiction  Puntaje_Selena  
2                     32         

Elegimos el modelo 3 para trabajar 